# 03 — Scorecard Model

This notebook trains the WOE logistic-regression model, evaluates ROC-AUC and KS, converts probability to a 0–1000 score, and saves a reusable scoring bundle.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_curve
from src.scorecard import (
    DEFAULT_FEATURES, ScoreParameters, build_bundle, evaluate, fit_scorecard,
    load_creditcard_csv, probability_to_score, restore_population_probability,
    save_bundle, split_data, transform_woe
)

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
FIGURE_DIR = PROJECT_ROOT / 'figures'
MODEL_PATH = PROJECT_ROOT / 'model' / 'scorecard_bundle.pkl'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = load_creditcard_csv(DATA_PATH)
X_train, X_test, y_train, y_test = split_data(df, test_size=0.30, random_state=42)
features = DEFAULT_FEATURES
model, definitions, iv_table = fit_scorecard(
    X_train, y_train, features=features, n_bins=10, alpha=0.5, random_state=42
)
X_test_woe = transform_woe(X_test, definitions)
evaluation = evaluate(model, X_test_woe, y_test)
print(f"ROC-AUC: {evaluation['roc_auc']:.4f}")
print(f"KS:      {evaluation['ks']:.4f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, evaluation['probability'])
plt.plot(fpr, tpr, label=f"AUC = {evaluation['roc_auc']:.4f}")
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'roc_curve.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
ks_values = tpr - fpr
ks_index = int(np.argmax(ks_values))
plt.plot(thresholds, tpr, label='Fraud cumulative rate')
plt.plot(thresholds, fpr, label='Normal cumulative rate')
plt.axvline(thresholds[ks_index], color='red', linestyle='--', label=f"KS={ks_values[ks_index]:.4f}")
plt.xlim(0, 1)
plt.xlabel('Probability Threshold')
plt.ylabel('Cumulative Rate')
plt.title('KS Curve')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'ks_curve.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
score_params = ScoreParameters(base_score=400, pdo=35, base_odds_good_to_bad=100)
population_bad_rate = float(y_train.mean())
calibrated_probability = restore_population_probability(
    evaluation['probability'], population_bad_rate
)
scores = probability_to_score(calibrated_probability, score_params)
score_summary = pd.Series(scores, name='Risk_Score').describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
score_summary

In [ ]:
plot_frame = pd.DataFrame({'Risk_Score': scores, 'Class': y_test.to_numpy()})
sns.histplot(data=plot_frame, x='Risk_Score', hue='Class', bins=60, stat='density', common_norm=False)
plt.title('Risk Score Distribution')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'risk_score_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
bands = [-1, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
band_frame = pd.DataFrame({'Risk_Score': scores, 'Class': y_test.to_numpy()})
band_frame['Score_Band'] = pd.cut(band_frame['Risk_Score'], bins=bands)
band_analysis = band_frame.groupby('Score_Band', observed=False)['Class'].agg(Transactions='count', Fraud_Count='sum')
band_analysis['Fraud_Rate'] = band_analysis['Fraud_Count'] / band_analysis['Transactions'].replace(0, np.nan) * 100
band_analysis.to_csv(OUTPUT_DIR / 'score_band_analysis.csv')
band_analysis

In [ ]:
bundle = build_bundle(
    model, definitions, features, score_params,
    population_bad_rate=population_bad_rate
)
save_bundle(bundle, MODEL_PATH)
metrics = {
    'roc_auc': evaluation['roc_auc'],
    'ks': evaluation['ks'],
    'population_bad_rate': population_bad_rate,
    'score_min': int(scores.min()),
    'score_median': float(np.median(scores)),
    'score_max': int(scores.max())
}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', MODEL_PATH)
metrics

The historical completed run reported ROC-AUC 0.9807 and KS 0.9364. Small differences can occur if package behavior or preprocessing changes; the saved `metrics.json` records the reproduced values.